In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/train_final.csv')
print("Shape:", df.shape)

Shape: (1460, 80)


In [2]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1168, 79)
Test size: (292, 79)


In [3]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=10),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': round(rmse, 4), 'R2': round(r2, 4)}
    print(f"{name} → RMSE: {rmse:.4f} | R²: {r2:.4f}")

Linear Regression → RMSE: 0.1530 | R²: 0.8746
Ridge → RMSE: 0.1525 | R²: 0.8753
XGBoost → RMSE: 0.1437 | R²: 0.8893


In [4]:
results_df = pd.DataFrame(results).T
print(results_df)

                     RMSE      R2
Linear Regression  0.1530  0.8746
Ridge              0.1525  0.8753
XGBoost            0.1437  0.8893


In [5]:
xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
cv_scores = cross_val_score(xgb, X, y, cv=5, scoring='r2')

print("Cross Validation R² Scores:", cv_scores)
print("Mean R²:", round(cv_scores.mean(), 4))
print("Std:", round(cv_scores.std(), 4))

Cross Validation R² Scores: [0.89797057 0.87592162 0.87536276 0.90014585 0.8756289 ]
Mean R²: 0.885
Std: 0.0115


In [6]:
import pickle

xgb.fit(X_train, y_train)

with open('../src/model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print("Model saved!")

Model saved!


In [2]:
import pickle
from sklearn.preprocessing import StandardScaler
import pandas as pd
df = pd.read_csv('../data/train_final.csv')
X = df.drop('SalePrice', axis=1)

scaler = StandardScaler()
scaler.fit(X)

with open('../src/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Scaler saved!")

Scaler saved!


In [3]:
df_unscaled = pd.read_csv('../data/train_cleaned.csv')
df_unscaled.to_csv('../data/train_unscaled.csv', index=False)
print("Unscaled data saved!")

Unscaled data saved!
